### This notebook will focus on finding the best model to predict the severity predictor !

In [3]:
### Libraries we will be using
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, accuracy_score,recall_score,f1_score,roc_curve,roc_auc_score, precision_recall_curve,confusion_matrix,ConfusionMatrixDisplay,classification_report, mean_absolute_error, cohen_kappa_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


In [4]:
## Data set we will be using 
data = pd.read_csv('../Datasets/pre-processedData.csv')
attack_data = data[data['is_malicious'] == 1]
attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)
attack_data.head(5)

/var/folders/bt/9_3kc80d4jq8sv8fv_c7d_fr0000gn/T/ipykernel_39868/3148040238.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)


,duration,src_bytes,dst_bytes,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,is_guest_login,count,srv_count,serror_rate,rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,level,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,Web_services,File_services,remote_login_service,email_service,dns_service,icmp_service,netbios_service,database_service,diagnostic_service,auth_service,messaging_service,other_service,flag_REJ,flag_RSTO,flag_RSTR,flag_S1,flag_S3,flag_SF,flag_SH,Severity_Score,bytes_ratio,total_bytes
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.820282,1.945910,1.0,0.0,0.05,0.07,0.0,255,26,0.10,0.05,0.0,0.0,1.0,0.0,0.0,19,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.804021,2.995732,0.0,1.0,0.16,0.06,0.0,255,19,0.07,0.07,0.0,0.0,0.0,1.0,1.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,2,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,5.117994,2.302585,1.0,0.0,0.05,0.06,0.0,255,9,0.04,0.05,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.770685,2.833213,1.0,0.0,0.14,0.06,0.0,255,15,0.06,0.07,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,5.602119,3.178054,1.0,0.0,0.09,0.05,0.0,255,23,0.09,0.05,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0.0,0.0


In [5]:
## Train test split of the data 
RANDOM_SEED = 42
X = attack_data.drop(columns=['Severity_Score'])
y = attack_data['Severity_Score']

## Splitting the data into train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=RANDOM_SEED)


---

### Model 1 : Logisitc Regression 

In [6]:
### Setting custom class weights based on the severity type 
class_weights = {1:1,2:3, 3:10} 
pipeline_steps = [('scaler', StandardScaler()),('logit', LogisticRegression(solver='lbfgs', class_weight = class_weights,C=1000,penalty='l2',max_iter = 1000))]
logit_pipeline = Pipeline(pipeline_steps)

logit_model = logit_pipeline.fit(X_train,y_train)

/opt/anaconda3/envs/Network_Intrusion/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [7]:
logit_training_pred = logit_model.predict(X_train)
logit_testing_pred = logit_model.predict(X_test)
logit_training_proba = logit_model.predict_proba(X_train)[:,1]
logit_testing_proba = logit_model.predict_proba(X_test)[:,1]


In [ ]:
## Evaluating the model
logit_mae = mean_absolute_error(y_test, logit_testing_pred)
print(f"{logit_mae:.3f}")

# Cohen Kappa Score 
cks = cohen_kappa_score(y_test, logit_testing_pred)
print(f"{cks:.2f}")


high_severe_testing_data = y_test == 3
high_severe_predicted_test_data = logit_testing_pred == 3

### Confusion Matrix for the severe cases 

true_positives = ((y_test == 3)& (logit_testing_pred == 3)).sum()
false_positives = ((y_test != 3) & (logit_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (logit_testing_pred != 3)).sum()
true_negatives = ((y_test != 3) & (logit_testing_pred != 3)).sum()



### Metrics for calculation 
logit_rare_precision = true_positives/(true_positives+false_positives) 
logit_rare_recall = true_positives/(true_positives + false_negatives)
logit_rare_fpr = false_positives/(true_negatives+false_positives) ## Misclassified
logit_rare_fnr = false_negatives/(true_positives+false_negatives) ## False Alaram




0.007
0.99
0.9756986634264885 0.9901356350184957 0.02466091245376079 0.0005933837709538644


---
### Model 2: Linear SVM 



In [26]:
pipeline_steps = [('scaler', StandardScaler()), ('svm', LinearSVC(multi_class='ovr', penalty='l2', C = 0.1))]
svm_pipeline = Pipeline(pipeline_steps)

